### **Rabin Cryptosystem**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/Rabin_Cryptosystem.ipynb)

> The **Rabin cryptosystem** encrypts by squaring: `c = m² mod n`, with `n = p · q`. Breaking it is provably as hard as **factoring** `n`. The catch: decryption yields **four square roots**, and the receiver must tell which one is the message. This notebook adds **redundancy** to the message so the right root is selected without knowing the original, a step the first version of this notebook skipped by comparing with the original message.

> [!WARNING]
> Educational code. Never use these implementations to protect real data: use a vetted library (for example, `cryptography`) with modern, authenticated algorithms.

In [1]:
# Primes with p ≡ q ≡ 3 (mod 4), which make square roots easy to compute
import secrets

def is_probable_prime(n, rounds=40):
    if n < 2:
        return False
    for small in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29):
        if n % small == 0:
            return n == small
    d, s = n - 1, 0
    while d % 2 == 0:
        d, s = d // 2, s + 1
    for _ in range(rounds):
        x = pow(secrets.randbelow(n - 3) + 2, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def blum_prime(bits):
    while True:
        candidate = secrets.randbits(bits) | (1 << (bits - 1)) | 3      # top bit set, ≡ 3 (mod 4)
        if is_probable_prime(candidate):
            return candidate

p, q = blum_prime(128), blum_prime(128)
n = p * q
print(f"p mod 4 = {p % 4}, q mod 4 = {q % 4}")
print(f"n = {n}  ({n.bit_length()} bits; real systems use 3072 bits or more)")

p mod 4 = 3, q mod 4 = 3
n = 50462701238610096865893349456745939557820314698647847594717579304118288263009  (255 bits; real systems use 3072 bits or more)


#### **01. Encryption and the Four Square Roots**

With `p ≡ 3 (mod 4)`, a square root of `c` modulo `p` is `c^((p+1)/4) mod p`. The roots modulo `p` and `q` are combined with the **Chinese Remainder Theorem (CRT)**, giving four candidates modulo `n`.

In [2]:
# Encryption, the four roots, and CRT
def encrypt(m):
    return pow(m, 2, n)

def four_roots(c):
    rp, rq = pow(c, (p + 1) // 4, p), pow(c, (q + 1) // 4, q)
    yp, yq = pow(p, -1, q), pow(q, -1, p)                 # modular inverses for the CRT
    roots = set()
    for a in (rp, p - rp):
        for b in (rq, q - rq):
            roots.add((a * q * yq + b * p * yp) % n)
    return sorted(roots)

message = int.from_bytes(b"Rabin demo", "big")
for root in four_roots(encrypt(message)):
    print(root, "<- message" if root == message else "")

389030477090634407964015 <- message
23378052541679985485123102030803776634816857870016977745269337306076788999873 
27084648696930111380770247425942162923003456828630869849448241998041499263136 
50462701238610096865893349456745939557820314698647847205687102213483880298994 


#### **02. Choosing the Right Root with Redundancy**

Before encrypting, the sender appends the **last 32 bits of the message again** (a simple redundancy pattern). Only the true root ends with a repeated pattern, so the receiver selects it **without knowing the original**. Standardized schemes use stronger padding for the same purpose.

In [3]:
# Encode with redundancy, decrypt, and select the root whose pattern matches
TAG_BITS = 32
MASK = (1 << TAG_BITS) - 1

def encode(text):
    m = int.from_bytes(text.encode(), "big")
    return (m << TAG_BITS) | (m & MASK)

def decode_and_select(c):
    valid = [r for r in four_roots(c) if (r & MASK) == ((r >> TAG_BITS) & MASK)]
    if len(valid) != 1:
        raise ValueError(f"{len(valid)} roots match the redundancy pattern")
    m = valid[0] >> TAG_BITS
    return m.to_bytes((m.bit_length() + 7) // 8, "big").decode()

for text in ["Rabin demo", "Factoring is hard", "2026"]:
    recovered = decode_and_select(encrypt(encode(text)))
    print(f"{text!r} -> {recovered!r}")
    assert recovered == text

'Rabin demo' -> 'Rabin demo'
'Factoring is hard' -> 'Factoring is hard'
'2026' -> '2026'


#### **Key Takeaways**

* Rabin is as hard to break as factoring `n`, a stronger guarantee than textbook RSA.
* Decryption is ambiguous (four roots); redundancy or padding resolves it.
* Use `secrets` for key generation, never `random`, and real key sizes (3072 bits or more).